# Render the full kiisafezone README showcase 🎬

Render the complete portrait composition from `assets/macos-readme-showcase.svg` using the Rust-based `svg-to-video` CLI. The source SVG is fetched from the latest `main` branch in this notebook.

**Target output:** 1080 × 1920 · 30 FPS · 59 seconds · H.264 MP4.

Run the cells from top to bottom. The renderer installation is needed only once per fresh Colab runtime.

In [ ]:
import os
import shutil
import subprocess
import urllib.request
from pathlib import Path

# Repair PATH without overwriting the system paths.
system_paths = ["/root/.cargo/bin", "/usr/local/sbin", "/usr/local/bin", "/usr/sbin", "/usr/bin", "/sbin", "/bin"]
existing_paths = os.environ.get("PATH", "").split(":")
os.environ["PATH"] = ":".join(dict.fromkeys(p for p in system_paths + existing_paths if p and Path(p).is_dir()))

def run(command, **kwargs):
    print("$", " ".join(map(str, command)))
    subprocess.run(command, check=True, **kwargs)

# Reuse anything already installed in this Colab runtime.
browser_names = ["google-chrome-stable", "google-chrome", "chromium", "chromium-browser", "chrome", "chrome-browser"]
browser = next((shutil.which(name) for name in browser_names if shutil.which(name)), None)
ffmpeg = shutil.which("ffmpeg")
cc = shutil.which("cc") or shutil.which("gcc")
cargo = shutil.which("cargo")
renderer = shutil.which("svg-to-video")

# Only use apt when a required system dependency is actually missing.
missing_packages = []
if not ffmpeg:
    missing_packages.append("ffmpeg")
if not cc:
    missing_packages.extend(["build-essential", "pkg-config", "libssl-dev"])
if not browser:
    missing_packages.append("ca-certificates")
if missing_packages:
    apt_get = shutil.which("apt-get")
    if not apt_get:
        raise RuntimeError("A required system dependency is missing and apt-get is unavailable. Use a hosted Linux Colab runtime.")
    run([apt_get, "update", "-qq"])
    run([apt_get, "install", "-y", "-qq", *list(dict.fromkeys(missing_packages))])
    ffmpeg = shutil.which("ffmpeg")
    cc = shutil.which("cc") or shutil.which("gcc")
    browser = next((shutil.which(name) for name in browser_names if shutil.which(name)), None)

# If Colab has no Chrome/Chromium, install official Google Chrome (not the Snap Chromium stub).
if not browser:
    apt_get = shutil.which("apt-get")
    chrome_deb = "/tmp/google-chrome-stable.deb"
    urllib.request.urlretrieve("https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb", chrome_deb)
    run([apt_get, "install", "-y", "-qq", chrome_deb])
    browser = shutil.which("google-chrome-stable")

# Install Rust only if this runtime does not already have Cargo.
if not cargo:
    rustup_script = urllib.request.urlopen("https://sh.rustup.rs").read().decode("utf-8")
    bash = shutil.which("bash")
    if not bash:
        raise RuntimeError("bash is unavailable; cannot install Rust.")
    run([bash, "-s", "--", "-y"], input=rustup_script, text=True)
    os.environ["PATH"] = "/root/.cargo/bin:" + os.environ["PATH"]
    cargo = shutil.which("cargo")

# Compile the renderer only when it is not already installed.
if not shutil.which("svg-to-video"):
    if not cargo:
        raise RuntimeError("Cargo is unavailable; svg-to-video cannot be installed.")
    run([cargo, "install", "svg-to-video", "--locked", "--jobs", "2"])
    renderer = shutil.which("svg-to-video")
else:
    renderer = shutil.which("svg-to-video")

for label, executable in [("Chrome/Chromium", browser), ("FFmpeg", shutil.which("ffmpeg")), ("Cargo", shutil.which("cargo")), ("svg-to-video", renderer)]:
    if not executable:
        raise RuntimeError(f"Required dependency is still missing: {label}")
    print(f"{label}: {executable}")

run([renderer, "--help"])
print("\nRenderer is ready. Existing dependencies were reused; only missing components were installed.")

In [ ]:
# Fetch the latest showcase source from the repository's main branch
%cd /content
!rm -rf /content/kiisafezone
!git clone --depth 1 --branch main https://github.com/kiisafezone/kiisafezone.git
%cd /content/kiisafezone
!git log -1 --oneline
!ls -lh assets/macos-readme-showcase.svg

In [ ]:
# Validate the actual source SVG before rendering
from pathlib import Path
import xml.etree.ElementTree as ET

svg_path = Path('/content/kiisafezone/assets/macos-readme-showcase.svg')
assert svg_path.is_file(), f'Missing source SVG: {svg_path}'
root = ET.parse(svg_path).getroot()
print('Source:', svg_path)
print('Canvas:', root.attrib.get('width'), '×', root.attrib.get('height'))
print('ViewBox:', root.attrib.get('viewBox'))
print('File size:', f'{svg_path.stat().st_size:,}', 'bytes')
assert root.attrib.get('width') == '1080', 'Unexpected SVG width'
assert root.attrib.get('height') == '1920', 'Unexpected SVG height'
assert root.attrib.get('viewBox') == '0 0 1080 1920', 'Unexpected SVG viewBox'
print('Preflight passed.')

In [ ]:
# Render the complete 59-second README showcase
%cd /content/kiisafezone
!mkdir -p out
!rm -f out/showcase.mp4
!svg-to-video video assets/macos-readme-showcase.svg out/showcase.mp4 --duration 59 --fps 30 --width 1080 --height 1920 --codec h264
!ls -lh out/showcase.mp4

In [ ]:
# Verify the rendered video's dimensions, frame rate, and duration
!ffprobe -v error -select_streams v:0 -show_entries stream=width,height,r_frame_rate -show_entries format=duration -of default=noprint_wrappers=1 out/showcase.mp4

In [ ]:
# Download the finished showcase MP4
from google.colab import files
files.download('/content/kiisafezone/out/showcase.mp4')